## Line Segment Intersection

A basic problem in many physical simulations is to determine how close points in 3D are to simple shapes.  This problem arises, for example, in robotics for collision detection.  In this lab, we consider a simple version of this problem where we build a Vitis HLS IP to compute the distance of 3D points to a line segment.

## Problem Formulation

The specific problem we consider is as follows.  We are given two points in 3D, `a` and `b`.  Hence, the line segement between the two points are the set of points:

```
z(t) = (1-t)*a + t*b,  t \in [0,1]
```

Then, we are given a set of points `x[i,:]`, `i=0,...,n-1` with each `x[i,:]` in 3D.  We wish to find:

```
dsq[i] = min_t ||x[i,:] - z(t)||^2
```

Pseudo-code for the algorithm to compute the distance square is as follows.

```python

dab = np.linalg.norm(b-a)  # norm ||b-a||
uab = (b-a) / dab   # unit vector in direction of the line segment

for i in range(n):
    w[i] = (x[i,:] - a).dot(uab)
    if (w[i] < 0):
        w[i] = 0
    elif (w[i] > dab):
        w[i] = dab
    z = a + w[i]*uab
    dsq[i] = np.sum((x[i,:] - z)**2)
```

Basically, the algorithm projects each point `x[i,:]` onto the line segement to find a point `z` on the line segement closest to `x[i,:]`. It then compute the distance.



## Constructing the Messages for the Interface Protocol

The first step in defining the IP that will implement the distance function is to define the command and response messages that go in and out of the IP.  We will use the following protocol.

- The processing system (PS) sends the IP a command header message `CmdHdr` with a transaction ID, the points `a` and `b`, the unit vector `uav`, and distance `dab`.  It also sends the number of samples `nsamp`.
Note that computing the `uav` and `dab` outside the IP is good since the IP does not need to perform division or square roots.
- The PS then sends an array `x` of shape `(nsamp, ndim)` where `ndim`.  This can be sent in a flattened form, namely as a vector of `nsamp*ndim` floating point numbers
- The IP first responds with a `RespHdr` message that echoes the transaction ID in the `CmdHdr`.  This ensures that the IP received the message.
- The IP then computes the distance squared and sends back the samples `dsq`.
- The IP finally sends a resppnse footer message, `RespFtr` message with the 

In [ ]:
from pysilicon.build.build import CodeGenConfig
from pysilicon.build.streamutils import copy_streamutils
from pysilicon.hw.arrayutils import gen_array_utils, read_array, get_nwords, write_uint32_file, read_uint32_file
from pysilicon.hw.dataschema import DataArray, DataList, EnumField, FloatField, IntField

import sys
from enum import IntEnum
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# Parameters
include_dir = "include"  # Directory for generated header files
word_bw_supported = [32, 64]  # Supported word bitwidths for the generated code
max_nsamp = 128  # Maximum number of samples for the input and output data arrays
ndim = 3  # Dimensionality of the points and line segments

# Element types
Float32 = FloatField.specialize(bitwidth=32, include_dir=include_dir)
TransId = IntField.specialize(bitwidth=16, signed=False, include_dir=include_dir)
Nsamp = IntField.specialize(bitwidth=16, signed=False, include_dir=include_dir)

# Error codes for intersection evaluation results
class IntersectError(IntEnum):
    NO_ERROR = 0
    TLAST_EARLY_CMD_HDR = 1  # TLAST was asserted before the full command header was received
    NO_TLAST_CMD_HDR = 2  # The full command header was received but TLAST was never asserted
    TLAST_EARLY_SAMP_IN = 3  # TLAST was asserted before all input samples were received
    NO_TLAST_SAMP_IN = 4  # All input samples were received but TLAST was never asserted
    WRONG_NSAMP = 5  # The number of samples received does not match the expected number
IntersectErrorField = EnumField.specialize(enum_type=IntersectError, include_dir=include_dir)


# Class for 3D points
# TODO:  Create a class for a point in 3D space that cab be used for the start and end points of the line segment, as well as the direction vector.  This class should be a DataArray with element type = Float32 and max_shape = (ndim,).
#   class Point(DataArray):
#     ... 


# Data structures for the command and response headers and footers
# TODO:  Create classes for the command header, response header, and response footer.  Think carefully about what fields should be included in
# each of these data structures for the protocol.  The points `a`, `b`, and unit vector `uab` should be included in the command header and use the `Point` class defined above.
# 
#   class CmdHdr(DataList):
#     ...
#   class RespHdr(DataList):
#     ...
#   class RespFtr(DataList):
#     ...

# List of all schema classes to be included in the generated code
schema_classes = [
    IntersectErrorField,
    Point,
    CmdHdr,
    RespHdr,
    RespFtr,
]

Next we generate the header files associated with the classes.  You should see a number of header files in the `include` directory.  

In [ ]:
# Make the include directory
include_dir = Path.cwd() / include_dir
include_dir.mkdir(exist_ok=True)

# Remove any existing header files in the include directory
for header_file in include_dir.glob("*.h"):
    header_file.unlink()

# Create a code generation configuration and generate the header files for all schema classes
cfg = CodeGenConfig(root_dir=Path.cwd(), util_dir=include_dir)

# Generate header files for all schema classes
for schema_class in schema_classes:
    out_path = schema_class.gen_include(cfg=cfg, word_bw_supported=word_bw_supported)
    print(f"generated {out_path}")

We also generate header files for general streaming utilities as well as streaming of floating point numbers

In [ ]:
# Generate the float32 include file with array utilities for the supported word bitwidths
out_path = gen_array_utils(Float32, word_bw_supported=word_bw_supported, cfg=cfg)
print(f"generated {out_path}")

# Copy the streamutils header file to the root directory
copy_streamutils(cfg=cfg)

## Building the Python Golden Model

Next build a python model for the IP.  We will write this now so that the inputs and outputs are the identical messages that the actual IP will receive from and transmit to the PS.

In [ ]:
import numpy.typing as npt
def line_inter(
    cmd_hdr: CmdHdr,
    x: npt.NDArray[np.float32]
) -> tuple[RespHdr, npt.NDArray[np.float32], RespFtr]:
    """ 
    Model for the IP to compute the squared distance from points to a line segment.

    Parameters
    ----------
    cmd_hdr : CmdHdr
        Command header containing the parameters for the intersection evaluation, including the start and end points of the line segment, the unit direction vector, and the number of samples.
    x : npt.NDArray[np.float32]
        Input array of shape (nsamp, ndim) containing the points for which to compute the squared distance to the line segment.  The number of samples (nsamp) should match the value specified in 
        the command header, and the number of dimensions (ndim) should match the dimensionality of the points (3 in this case).

    Returns
    -------
    resp_hdr : RespHdr
        Response header containing the transaction ID echoed from the command header.
    dsq : npt.NDArray[np.float32]
        Output array of shape (nsamp,) containing the squared distances from each input point to the line segment defined by the command header parameters.
    resp_ftr : RespFtr
        Response footer containing the number of elements read in the response data array and an error code indicating
    """

    # TODO:  Compute the outputs from the inputs


    return resp_hdr, dsq, resp_ftr


## Creating the Test Vectors

We next build the test vectors that we will test the model on.  We take a line segment `[a,b]` and a set of points `x` along some path.

In [ ]:
import numpy as np
a = np.array([-10, -10, -10])
b = np.array([10, 10, 10])

xstart = np.array([-10,10,30])
xend = np.array([10,15,-30])
nx = 100
tx = np.linspace(0, 1, nx)
x = xstart + tx[:, None] * (xend - xstart)

Create a 3D plot of the using `ax.plot(...)` that shows:
- The set of points `x[i,:]`
- THe line segment 

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')


# TODO:  Plot the points `x` as well as the line segment defined by points `a` and `b`.

# Print the plot
plot_dir = Path.cwd() / "plots"
plot_dir.mkdir(exist_ok=True)
plt.savefig(plot_dir / "line_inter_plot.png")

Nest we run the data through the IP and plot the distance squared vs. time

In [ ]:

# TODO:  Create the command header
#   cmd_hdr = CmdHdr()
#   cmd_hdr.a = a 
#   ...


# Call the intersect function
resp_hdr, dsq, resp_ftr = line_inter(cmd_hdr, x)

# Plot the squared distances
plt.plot(tx, dsq)
plt.grid()
plt.xlabel('Time t')
plt.ylabel('Squared Distance')

# Save the plot
plt.savefig(plot_dir / "dsq_plot.png")

Now we write the test vectors to files that can be read in the test bench.

In [ ]:

data_dir = Path.cwd() / "data"
data_dir.mkdir(parents=True, exist_ok=True)
cmd_hdr.write_uint32_file(data_dir / "cmd_hdr_data.bin")
write_uint32_file(x.flatten(), elem_type=Float32, file_path=data_dir / "x_in_data.bin", 
                  nwrite=cmd_hdr.nsamp * ndim)

write_uint32_file(dsq, elem_type=Float32, file_path=data_dir / "dsq_exp.bin")


We create a CSV file of the values `x` and `dsq` that will be used for the auto-grading.

In [ ]:
# Create a CSV file with the columns x[:,0],... x[:,2], dsq.  This will be used for the auto-gradiing
import pandas as pd
df = pd.DataFrame({
    "x0": x[:,0],
    "x1": x[:,1],
    "x2": x[:,2],
    "dsq": dsq
})
df.to_csv(data_dir / "dsq_python.csv", index=False)

## Implementing the Kernel and Testbench

Now go to `line_inter.cpp` and complete all the sections marked `TODO`.

## C-Simulation and Functional Verification

When the kernel is completed implementation, you can run the C simulation.  You do this in the command line as follows:

- Follow the [instructions](https://sdrangan.github.io/hwdesign/docs/support/amd/lauching.html) to set the path for the Vitis command line tools.  This will work in Windows or Linux including the NYU EDA server.
- In the command terminal, run:

```bash
vitis-run --mode hls --tcl run_hls.tcl
```

If you prefer, you can also run the C simulation from the GUI.

During the C simulation, you should see a print out of `MAE`, the mean absolute error. If your design is correct, the MAE should be about `1e-5`.


In [ ]:
print('Response header fields:')
resp_hdr_dut = RespHdr()
resp_hdr_dut.read_uint32_file(data_dir / "resp_hdr_data.bin")
for k in resp_hdr_dut.elements:
    print(f"{k}: {getattr(resp_hdr_dut, k)}")

print('\nResponse footer fields:')
resp_ftr_dut = RespFtr()
resp_ftr_dut.read_uint32_file(data_dir / "resp_ftr_data.bin")
for k in resp_ftr_dut.elements:
    v = getattr(resp_ftr_dut, k)
    if k == "error":
        v = v.name
    print(f"{k}: {v}")



Next, we read the values from of the DUT

In [ ]:
# Read the values from the DUT output file
dsq_dut = read_uint32_file(data_dir / "dsq_dut.bin", elem_type=Float32, shape=(cmd_hdr.nsamp,))

# TODO:  Plot the DUT output values along with the expected values, and compute the mean absolute error (MAE) between the DUT output and the expected values.


# Compute the mean absolute error (MAE).  THe value should be around 1e-5
mae = np.mean(np.abs(dsq_dut - dsq))
print(f"MAE: {mae}")

Finally, we create a CSV file of the results that will be used for verification

In [ ]:
# Create a CSV file with the columns x[:,0],... x[:,2], dsq, dsq_dut.  This will be used for the auto-gradiing
import pandas as pd
df = pd.DataFrame({
    "x0": x[:,0],
    "x1": x[:,1],
    "x2": x[:,2],
    "dsq": dsq,
    "dsq_dut": dsq_dut,
})
df.to_csv(data_dir / "dsq_comparison.csv", index=False)

## Reading the pipelining report

Once the functional test is working, you can validate that the pipeling is working.  The C synthesis produces an XML report.  You can locate it in 

```bash
hls_component/solution1/syn/reports/csynth.xml
```

We will use the PySilicon utilities to parse this report.  You should see that all loops obtain a PipelineII = 1.


In [ ]:
from pysilicon.utils.csynthparse import CsynthParser
import pandas as pd
import os

# Parse the synthesis for UF=1 
sol_path = os.path.join(os.getcwd(), 'hls_component', 'solution1')
parser = CsynthParser(sol_path=sol_path)

# Get the latency and initiation interval
print('Latency and Initiation Interval:')
parser.get_loop_pipeline_info()
#print(parser.loop_df)
with pd.option_context("display.max_columns", None, "display.width", 300):
    print(parser.loop_df)


# Get the resources
print('\nResource Usage:')
parser.get_resources()
print(parser.res_df)

In [ ]:
# Write the parser.loop_df to a CSV file in the data directory for auto-grading
parser.loop_df.to_csv(data_dir / "csynth_loop_info.csv", index=False)

# Write the resource usage dataframe to a CSV file in the data directory for auto-grading
parser.res_df.to_csv(data_dir / "csynth_resource_usage.csv", index=False)